# US Flight Delays Network Analysis
## Step 1: Data Ingestion & Cleaning

**Course:** CS-GY 6513-C Big Data — Spring 2026

**Team:** Mirsaid Abbasov [ma9197], Nicholas Pesa [np2354], Ferdi Fadillah [ff2364]

---

This notebook handles the first stage of our pipeline:
1. Install PySpark and mount Google Drive
2. Load all 60 monthly BTS CSV files (Jan 2019 – Dec 2023)
3. Clean and preprocess the data
4. Save as partitioned Parquet files for efficient downstream analysis

## 1. Environment Setup

In [ ]:
!pip install pyspark -q

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, DoubleType, StringType, DateType

spark = SparkSession.builder \
    .appName("FlightDelays_Step1") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

print("Spark session created successfully")
print(f"Spark version: {spark.version}")

Spark session created successfully
Spark version: 4.0.2


## 2. Data Ingestion

Loading all 60 monthly BTS TranStats CSV files from Google Drive.

**Source:** [BTS Reporting Carrier On-Time Performance](https://www.transtats.bts.gov/DL_SelectFields.aspx)

**Date range:** January 2019 – December 2023

In [ ]:
# Path to your CSV files in Google Drive
DATA_PATH = "/content/drive/MyDrive/Big Data Project/*.csv"

# Read all CSVs at once
raw_df = spark.read.csv(
    DATA_PATH,
    header=True,
    inferSchema=True
)

print(f"Total raw records: {raw_df.count():,}")
print(f"Number of columns: {len(raw_df.columns)}")

Total raw records: 31,682,812
Number of columns: 22


In [ ]:
# Quick look at the schema
raw_df.printSchema()

root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: string (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DEL15: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double (nullable = true)
 |-- NAS_DELAY: double (nullable = true)
 |-- SECURITY_DELAY: double (nullable = true)
 |-- LATE_AIRCRAFT_DELAY: double (nullable = true)



In [ ]:
# Preview first 5 rows
raw_df.show(5, truncate=False)

+----+-----+-----------+--------------------+-----------------+-----------------+------+---------------+----+------------+---------+------------+---------+---------+---------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|YEAR|MONTH|DAY_OF_WEEK|FL_DATE             |OP_UNIQUE_CARRIER|ORIGIN_AIRPORT_ID|ORIGIN|DEST_AIRPORT_ID|DEST|CRS_DEP_TIME|DEP_DELAY|CRS_ARR_TIME|ARR_DELAY|ARR_DEL15|CANCELLED|DIVERTED|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|
+----+-----+-----------+--------------------+-----------------+-----------------+------+---------------+----+------------+---------+------------+---------+---------+---------+--------+--------+-------------+-------------+---------+--------------+-------------------+
|2019|7    |1          |7/1/2019 12:00:00 AM|9E               |10135            |ABE   |10397          |ATL |1459        |-1.0     |1710        |-21.0    |0.0      |0.0      |0.0     |692.0   |NULL  

## 3. Data Cleaning

Steps:
- Drop any unnamed/empty columns that BTS sometimes adds
- Cast columns to correct types
- Separate cancelled/diverted flights
- Handle missing values in delay fields

In [ ]:
# BTS CSVs sometimes have a trailing empty column — drop any unnamed columns
columns_to_keep = [c for c in raw_df.columns if not c.startswith('_c') and c.strip() != '']
df = raw_df.select(columns_to_keep)

print(f"Columns after cleanup: {df.columns}")

Columns after cleanup: ['YEAR', 'MONTH', 'DAY_OF_WEEK', 'FL_DATE', 'OP_UNIQUE_CARRIER', 'ORIGIN_AIRPORT_ID', 'ORIGIN', 'DEST_AIRPORT_ID', 'DEST', 'CRS_DEP_TIME', 'DEP_DELAY', 'CRS_ARR_TIME', 'ARR_DELAY', 'ARR_DEL15', 'CANCELLED', 'DIVERTED', 'DISTANCE', 'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY', 'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']


In [ ]:
# Define expected columns and their types
expected_columns = [
    'YEAR', 'MONTH', 'DAY_OF_WEEK', 'FL_DATE',
    'OP_UNIQUE_CARRIER',
    'ORIGIN_AIRPORT_ID', 'ORIGIN',
    'DEST_AIRPORT_ID', 'DEST',
    'CRS_DEP_TIME', 'DEP_DELAY',
    'CRS_ARR_TIME', 'ARR_DELAY', 'ARR_DEL15',
    'CANCELLED', 'DIVERTED',
    'DISTANCE',
    'CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
    'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY'
]

# Check which expected columns are present
present = [c for c in expected_columns if c in df.columns]
missing = [c for c in expected_columns if c not in df.columns]

print(f"Present: {len(present)}/{len(expected_columns)}")
if missing:
    print(f"Missing columns: {missing}")
else:
    print("All expected columns found!")

Present: 22/22
All expected columns found!


In [ ]:
# Select only expected columns
df = df.select(present)

In [ ]:
from pyspark.sql import functions as F

# Cast columns to proper types
# We split by space to take only 'M/d/yyyy' and ignore '12:00:00 AM'
df = df \
    .withColumn('YEAR', F.col('YEAR').cast(IntegerType())) \
    .withColumn('MONTH', F.col('MONTH').cast(IntegerType())) \
    .withColumn('DAY_OF_WEEK', F.col('DAY_OF_WEEK').cast(IntegerType())) \
    .withColumn('FL_DATE', F.to_date(F.split(F.col('FL_DATE'), ' ').getItem(0), 'M/d/yyyy')) \
    .withColumn('ORIGIN_AIRPORT_ID', F.col('ORIGIN_AIRPORT_ID').cast(IntegerType())) \
    .withColumn('DEST_AIRPORT_ID', F.col('DEST_AIRPORT_ID').cast(IntegerType())) \
    .withColumn('CRS_DEP_TIME', F.col('CRS_DEP_TIME').cast(IntegerType())) \
    .withColumn('DEP_DELAY', F.col('DEP_DELAY').cast(DoubleType())) \
    .withColumn('CRS_ARR_TIME', F.col('CRS_ARR_TIME').cast(IntegerType())) \
    .withColumn('ARR_DELAY', F.col('ARR_DELAY').cast(DoubleType())) \
    .withColumn('ARR_DEL15', F.col('ARR_DEL15').cast(DoubleType())) \
    .withColumn('CANCELLED', F.col('CANCELLED').cast(DoubleType())) \
    .withColumn('DIVERTED', F.col('DIVERTED').cast(DoubleType())) \
    .withColumn('DISTANCE', F.col('DISTANCE').cast(DoubleType())) \
    .withColumn('CARRIER_DELAY', F.col('CARRIER_DELAY').cast(DoubleType())) \
    .withColumn('WEATHER_DELAY', F.col('WEATHER_DELAY').cast(DoubleType())) \
    .withColumn('NAS_DELAY', F.col('NAS_DELAY').cast(DoubleType())) \
    .withColumn('SECURITY_DELAY', F.col('SECURITY_DELAY').cast(DoubleType())) \
    .withColumn('LATE_AIRCRAFT_DELAY', F.col('LATE_AIRCRAFT_DELAY').cast(DoubleType()))

print("Columns cast to correct types. Please re-run the filtering cells (3.2 and 3.3) before saving.")
df.printSchema()

Columns cast to correct types. Please re-run the filtering cells (3.2 and 3.3) before saving.
root
 |-- YEAR: integer (nullable = true)
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- FL_DATE: date (nullable = true)
 |-- OP_UNIQUE_CARRIER: string (nullable = true)
 |-- ORIGIN_AIRPORT_ID: integer (nullable = true)
 |-- ORIGIN: string (nullable = true)
 |-- DEST_AIRPORT_ID: integer (nullable = true)
 |-- DEST: string (nullable = true)
 |-- CRS_DEP_TIME: integer (nullable = true)
 |-- DEP_DELAY: double (nullable = true)
 |-- CRS_ARR_TIME: integer (nullable = true)
 |-- ARR_DELAY: double (nullable = true)
 |-- ARR_DEL15: double (nullable = true)
 |-- CANCELLED: double (nullable = true)
 |-- DIVERTED: double (nullable = true)
 |-- DISTANCE: double (nullable = true)
 |-- CARRIER_DELAY: double (nullable = true)
 |-- WEATHER_DELAY: double (nullable = true)
 |-- NAS_DELAY: double (nullable = true)
 |-- SECURITY_DELAY: double (nullable = true)
 |-- LATE_AIR

### 3.1 Check for Nulls

In [ ]:
# Count nulls per column
# Using a simple check to avoid parser exceptions on the existing data
null_counts = df.select(
    [F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in df.columns]
)

display(null_counts)

DataFrame[YEAR: bigint, MONTH: bigint, DAY_OF_WEEK: bigint, FL_DATE: bigint, OP_UNIQUE_CARRIER: bigint, ORIGIN_AIRPORT_ID: bigint, ORIGIN: bigint, DEST_AIRPORT_ID: bigint, DEST: bigint, CRS_DEP_TIME: bigint, DEP_DELAY: bigint, CRS_ARR_TIME: bigint, ARR_DELAY: bigint, ARR_DEL15: bigint, CANCELLED: bigint, DIVERTED: bigint, DISTANCE: bigint, CARRIER_DELAY: bigint, WEATHER_DELAY: bigint, NAS_DELAY: bigint, SECURITY_DELAY: bigint, LATE_AIRCRAFT_DELAY: bigint]

### 3.2 Separate Cancelled & Diverted Flights

Cancelled and diverted flights don't have meaningful delay data, so we separate them out. We keep them for reference but our main analysis uses completed flights only.

In [ ]:
total = df.count()
cancelled = df.filter(F.col('CANCELLED') == 1.0)
diverted = df.filter(F.col('DIVERTED') == 1.0)
completed = df.filter((F.col('CANCELLED') == 0.0) & (F.col('DIVERTED') == 0.0))

print(f"Total flights:     {total:,}")
print(f"Cancelled:         {cancelled.count():,}")
print(f"Diverted:          {diverted.count():,}")
print(f"Completed:         {completed.count():,}")

Total flights:     31,682,812
Cancelled:         788,248
Diverted:          73,119
Completed:         30,821,445


### 3.3 Handle Missing Delay Values

For completed flights, missing delay cause fields (CarrierDelay, WeatherDelay, etc.) mean there was no delay of that type — so we fill them with 0. Missing DepDelay/ArrDelay on completed flights are rare edge cases that we drop.

In [ ]:
# Fill delay cause columns with 0 (no delay = 0)
delay_cause_cols = ['CARRIER_DELAY', 'WEATHER_DELAY', 'NAS_DELAY',
                    'SECURITY_DELAY', 'LATE_AIRCRAFT_DELAY']

completed = completed.fillna(0, subset=delay_cause_cols)

# Drop rows where core delay values are null (rare edge cases)
before = completed.count()
completed = completed.dropna(subset=['DEP_DELAY', 'ARR_DELAY', 'ARR_DEL15'])
after = completed.count()

print(f"Dropped {before - after:,} rows with missing core delay values")
print(f"Clean completed flights: {after:,}")

Dropped 4 rows with missing core delay values
Clean completed flights: 30,821,441


## 4. Quick Sanity Checks

In [ ]:
# Verify year range
completed.groupBy('YEAR').count().orderBy('YEAR').show()

+----+-------+
|YEAR|  count|
+----+-------+
|2019|7268232|
|2020|4399575|
|2021|5878219|
|2022|6532012|
|2023|6743403|
+----+-------+



In [ ]:
# Verify month distribution for one year
completed.filter(F.col('YEAR') == 2023).groupBy('MONTH').count().orderBy('MONTH').show(12)

+-----+------+
|MONTH| count|
+-----+------+
|    1|527197|
|    2|492747|
|    3|571533|
|    4|550249|
|    5|575429|
|    6|562804|
|    7|585058|
|    8|592142|
|    9|560887|
|   10|596003|
|   11|562413|
|   12|566941|
+-----+------+



In [ ]:
# Basic delay stats
completed.select(
    F.mean('DEP_DELAY').alias('avg_dep_delay'),
    F.mean('ARR_DELAY').alias('avg_arr_delay'),
    F.mean('ARR_DEL15').alias('pct_delayed_15min'),
    F.min('DEP_DELAY').alias('min_dep_delay'),
    F.max('DEP_DELAY').alias('max_dep_delay')
).show(truncate=False)

+----------------+-----------------+-------------------+-------------+-------------+
|avg_dep_delay   |avg_arr_delay    |pct_delayed_15min  |min_dep_delay|max_dep_delay|
+----------------+-----------------+-------------------+-------------+-------------+
|9.95142981796341|4.095112976709947|0.18152957222214236|-128.0       |4413.0       |
+----------------+-----------------+-------------------+-------------+-------------+



In [ ]:
# Number of unique airports and carriers
n_origins = completed.select('ORIGIN').distinct().count()
n_dests = completed.select('DEST').distinct().count()
n_carriers = completed.select('OP_UNIQUE_CARRIER').distinct().count()
n_routes = completed.select('ORIGIN', 'DEST').distinct().count()

print(f"Unique origin airports: {n_origins}")
print(f"Unique dest airports:   {n_dests}")
print(f"Unique carriers:        {n_carriers}")
print(f"Unique routes:          {n_routes}")

Unique origin airports: 380
Unique dest airports:   381
Unique carriers:        18
Unique routes:          8128


## 5. Save as Partitioned Parquet

We save the cleaned completed flights as Parquet files partitioned by YEAR and MONTH. This makes downstream Spark queries much faster since Spark can skip irrelevant partitions.

In [ ]:
# Output path in Google Drive
OUTPUT_PATH = "/content/drive/MyDrive/Big Data Project/parquet/completed_flights"

# Ensure we are using the cleaned 'completed' dataframe
completed.write \
    .partitionBy('YEAR', 'MONTH') \
    .mode('overwrite') \
    .parquet(OUTPUT_PATH)

print(f"Saved cleaned data to: {OUTPUT_PATH}")

Saved cleaned data to: /content/drive/MyDrive/Big Data Project/parquet/completed_flights


In [ ]:
# Also save the cancelled/diverted flights separately (for reference)
CANCELLED_PATH = "/content/drive/MyDrive/Big Data Project/parquet/cancelled_flights"

cancelled_diverted = df.filter((F.col('CANCELLED') == 1.0) | (F.col('DIVERTED') == 1.0))
cancelled_diverted.write \
    .partitionBy('YEAR', 'MONTH') \
    .mode('overwrite') \
    .parquet(CANCELLED_PATH)

print(f"Saved cancelled/diverted data to: {CANCELLED_PATH}")

Saved cancelled/diverted data to: /content/drive/MyDrive/Big Data Project/parquet/cancelled_flights


## 6. Verify Parquet Output

In [ ]:
# Read back and verify
verify_df = spark.read.parquet(OUTPUT_PATH)
print(f"Parquet row count: {verify_df.count():,}")
print(f"Partitions: {verify_df.select('YEAR', 'MONTH').distinct().count()} year-month combos")
verify_df.show(3)

Parquet row count: 30,821,441
Partitions: 60 year-month combos
+-----------+----------+-----------------+-----------------+------+---------------+----+------------+---------+------------+---------+---------+---------+--------+--------+-------------+-------------+---------+--------------+-------------------+----+-----+
|DAY_OF_WEEK|   FL_DATE|OP_UNIQUE_CARRIER|ORIGIN_AIRPORT_ID|ORIGIN|DEST_AIRPORT_ID|DEST|CRS_DEP_TIME|DEP_DELAY|CRS_ARR_TIME|ARR_DELAY|ARR_DEL15|CANCELLED|DIVERTED|DISTANCE|CARRIER_DELAY|WEATHER_DELAY|NAS_DELAY|SECURITY_DELAY|LATE_AIRCRAFT_DELAY|YEAR|MONTH|
+-----------+----------+-----------------+-----------------+------+---------------+----+------------+---------+------------+---------+---------+---------+--------+--------+-------------+-------------+---------+--------------+-------------------+----+-----+
|          1|2019-07-01|               9E|            10135|   ABE|          10397| ATL|        1459|     -1.0|        1710|    -21.0|      0.0|      0.0|     0.0|   

## Summary

**What we did in this step:**
- Loaded 60 monthly BTS CSV files (Jan 2019 – Dec 2023) into PySpark
- Cleaned column types and removed trailing empty columns
- Separated cancelled/diverted flights from completed flights
- Filled missing delay cause values with 0
- Dropped rare rows with missing core delay data
- Saved cleaned data as partitioned Parquet (by year and month)

**Next step:** Graph construction — building the airport-route network with delay-based edge weights.

In [ ]:
spark.stop()
print("Spark session stopped. Step 1 complete!")

Spark session stopped. Step 1 complete!
